FINE-TUNING DI BERT:
DALL'INTELLIGENZA GENERALE ALLA SPECIALIZZAZIONE

E' il processo in cui prendi un modello BERT già preaddestrato sul linguaggio generale e lo specializzi per un compito preciso, per esempio sentiment analysis, classificazioni mail, NER o spam detecion
Quindi non parti da zero, questo è il punto fondamentale
Durante il pre-training BERT ha già imparato moltissimo sul linguaggio. Per esempio ha imparato rappresentazione contestuali grazie soprattutto al Masked Languade Modeling
Ma attenzioe: questo BERT generale non è ancora necessarimente un classificatore di sentimen
Non sa che tu vuoi: 0=negativo 1=positivo
Per questo serve fine-tuning

Quando inizi BERT preaddestrato i suoi pesi NON sono casuali. Sono già il risultato del pretraining.
Durante il fine-tuning fai:
testo -> BERT -> prediction -> confronto con etichetta vera -> loss -> backpropagation
La backpropagation modifica i pesi in modo che il modello diventi più adatto al tuo task
Questo processo è ripetuto su miloni di esempi
La cosa importante è che, nel fine-tuning classico di BERT, vengono normalmente aggiornati anche i pesi del Transformer, non soltanto quelli del classificatore finale.

Il dataset di fine-tuning dipende dal task

Un altro punto importante riguarda il learning rate.
Durante il fine-tuning BERT normalmente si usano learning rate piuttosto piccoli rispetto a molti modelli addestrati da zero.
Il motivo è intuitivo: hai già pesi buoni che arrivano dal pretraining non vuoi stravolgere tutto ciò che BERT ha imparato, vuoi piuttosto specializzazione, piccoli aggiustamento

C'è un doppio utilizzo di BERT:
1) Feature Extraction
2) Fine-Tuning

1) - Feature Extraction 
BERT congeltao -> estraggo rappresentazionoi -> addestro solo il classificatore
In questo caso i pesi di BERT non cambiano
Ho quindi un modello preaddestrato usato come estrattore di feature

2) - Fine Tunning 
BERT + classification head + training
e aggiorni anche BERT
I Pesi di bert vengono midificati, spacilizzandolo al nostro contesto
Esistono anche tecniche intermedie: puoi congelare alcuni layer e sbloccarne altri
L'idea è che gli strati più bassi possono contenre rappresentazioni linguistiche più generiche, mentre quelli superiori possono essere maggiormente adatti al task
Oggi esistono anche tecniche di parameter-efficient fine-tunning, come adapter e LoRA, che modificano o aggiungono una quantità molto più piccola di parametri rispetto al fine-tuning completo

Poi c'è anche il PROMPTING
Con il prompting cambio l'istruzione ma i pesi rimangono invariati

Il fine-tuning di BERT consiste nel partire dai pesi di un Transformer già preaddestrato e continuare l'addestramento su un dataset specifico del task, generalmente aggiungendo una testa di output appropriata, in modo da adattare le rappresentazioni linguistiche generali al problema da risolvere.

Immagina di avere un laureato con il massimo dei voti in lettere, questo è BERT con il suo preaddestramento.
Tuttavia, il laureato, non conosce nulla del nostro ambiente (es. pratiche legali) e deve imparare questo nuovo contesto.
Partendo dalla sua conoscenza generale portandolo verso un'intelligenza verticale.

** HUGGING FACE **

Hugging Face e la Libreria Transformers
Huggin Face il portale che ha democratizzato il linguaggio naturale
Lo standard de facto per il NLP moderno
Se volessimo costruire un modello come BERT da zero, ci servirebbero mesi di calcolo e tanti tanti soldi. 
Fortunatamente la libreria Transformers di Hugging Face ci permette di saltare questa fase.
Come avere accesso ad un magazzino infinito di motori già collaudati.
Utilizzando classi come 'BertTokenizer' e 'BertModel', possiamo gestire l'intera pipeline di preprocessing e inferenza in modo modulare e scalabile.

Hugging Face non è un modello AI, è un ecositema molto ampio per trovare, scaricare, usare, addestrare, fine-tuning, valutare e condividere modelli di Machine Learning e AI. Oggi il suo Hub ospita modelli, dataset, e applicazioni demo chiamate Spaces.

La parte più conosciuta è Hugging Face Hub. Puoi immaginarlo come un 'GitHub' dei modelli AI: trovi repository contenenti pesi, configurazini, tokenizer, Model Card, informazioni sulla licenza e spesso risultati di valutazione. Il Hub ospita modelli per NLP, visione, audio e altri task.
Sull'Hub puoi trovare: BERT, DistlBERT, T5, Qwen, Llama, modelli per immagini, modelli per audio.
Hugging Face normalmente non ha creato questi modelli, Huggin Face è il luovo attraverso cui accedi al modello, non chi ha inventato il modello.

Il secondo componente più conosciuto è il Transformers. E' una libreria Python mantenuta da Hugging Face e dalla comunità che supporta modelli preaddestrati per testo, visione, audio e funziona con framework come PyTorch, TensorFlow, JAX

Con from_pretrained(..) non devi costruire manulmente BERT (o altro) layer per layer, ma Hugging Face recupera: architettura + configurazione + pesi preaddestrati + tokenizer corretto.
La documentazione delle AutoClasses specifica proprio che AutoModel e AutoTokenizer determinano automaticamente l'architettura appropriata a partire dal modello scelto.

Un'altra cosa che puoi fare con Huggin Face, oltre ad usare il modello originale, è modificarlo tramite fine-tuning. La libreria Transformes mette a disposizione strumenti per farlo, incluso Trainer.

Un'altra cosa interessante è Datasets, puoi caricare datasets già preparati per Machine Learning/NLP

Un'altra cose importante è PEFT (Parameter-Efficinet Fine-Tuning) permette di addestrare solo una piccola parte aggiuntiva del modello scelto, riducendo fortementi costi di memoria e calcolo. Uno dei metodi più famosi è LoRA che mantiene congelati i pesi originali e apprende matrici aggiuntive molto più piccole riducendo il numero di parametri da addestare. questo diventa importante con Llama, Qwen ed altri LLM grandi

Componenti dell'Ecositema
Strumenti per il caricamente e la configurazione
L'ecostitema si basa su 4 pilastri:
- AtuoModel: classe intelligente che capisce da sola quale motore state caricando semplicemente leggendo il nome del checkpoint (es. 'bert-base-uncased') 
- Tokenizer che è il nostro traduttore, prende il testo umano e lo riduce in frammenti, scomponendo il testo in sub-words (WordPiece), che la rete può digerire e aggiunge i token speciali necessari
- Config: il manuale d'uso del modello che ci dice quanto layer ci sono e quanto è profonda l'attenzione
- Hub: una sorta di piazza del mercato globale dove i ricercatori di tutto il mondo condividono modelli addestrati su ogni lingua possibile e diversi domini.

Vediamo come questi pezzi si incastrano in un Wordflow reale.

Workflow di Caricamento
Il processo inizia con il metodo fram_pretrained
- Download dei pesi: il metodo from_pretrained() scarica i pesi, li mette in cache ed inizializza la memoria, assicurando che il modello sia pronto per l'uso immediato. Come scaricare un cervello preaddestrato.
- Preprocessing Integrato: il tokenizer non si limita a dividere le parole, genera una mappa completa per la rete: - 'input_ids': sono i codici delle parole , 'attention_mask': dice alla rete dove guardare e cosa ignorare e 'token_type_id' richiesti da BERT
- Integrazione con TensorFlow: tutto questo è integrato con TensorFlow e Keras, permettendoci di usare BERT come se fosse un qualsiasi layer di una rete neurale classica.

Efficienza del Tokenizer
Il tokenizer deve essere efficiente perchè la reta ha bisogno di sequenze di lunghezza fissa, se una frase è troppo corta usiamo il padding (ovvero degli 0) ma non vogliamo che la rete sprechi energia leggendo gli zeri, ecco perchè diventa importante la maschera di attenzione (atention mask) che è fondamentale per indicare alla rete quali token ignorare (padding)
Questo permette a BERT di focalizzarsi solo sull'informazione del messaggio.

Una volta preparati i dati, dobbiamo decidere come modificare il modello per il nostro compito

Adattamento a Task Specifici
Dall'intelligenza generale alla specializzazione
Il fine-tuning consiste nel prendere BERT (il 'corpo' del modello) e aggiungere una 'testa' (un layer lineare) adatta al nostro problema specifico. Manteniamo il corpo intatto e modifichiamo la testa.
Durante l'addestramento, aggiorniamo leggeremente i pesi di BERT e addestriamo da zero la nuova testa per mappare le feature sui nostri target.

Ma quali sono le regolo d'oro per non rovinare il modello durante questo processo?

Strategia di Fine-tuning
Ottimizzazione del Transfer Learning
La strategia è fondamentale, poichè BERT sa già molto se usassimo un passo di apprendimento troppo alto (learning rate) sarebbe come usare una ruspa e distruggere la conoscenza pre-acquisita di BERT.
Dobbiamo usare un lr ridottissimo, possiamo anche scegliere il Frezzing, congeliamo i primi layer del modello, quelli che conoscono le basi della grammatica, ed addestriamo solo la parte finale.
E' come dire ad uno studente, so che sai già leggere, ora concentrati solo sui termini legati a questo particolare argomento.

Configurazione della Perdita
Il nostro termometro per capire se stiamo facendo bene, è la funzione di perdita.
- Loss per Classificazione: per la classificazione binaria o multi-classe, utilizziamo solitamente la Cross-Entropy applicati ai logit delle testa finale
- Adattamente dei Logit: BERT restituisce punteggi numerici grezzi (logit), è compito nostro applicare 'softmax' o 'sigmoid' a seconda del numero di classi. questo trasforma i punteggi restituiti da BERT in probabilità che noi possiamo capire.
- Metriche di Monitoraggio: ma non guardare solo l'Accuracy, in NLP, specialmente su datasets sbilanciati, l'accuratezza può mentire, dobbiamo monitorare la f1-score, per essere sicuri che il modello non stia solo tirando ad indovinare la classe più frequente.

Aggiornamento dei Pesi
Per il fine-tuning usiamo un ottimizzatore speciale che gestisca correttamente il decadimento dei pesi per preservare la struttura semantica del modello pre-addestrato. Questo ottimizzatore è AdamW
A differenza rispetto as Adam classico è come gestisce il decadimento dei pesi. vogliamo che i pesi cambiano, si, ma non vogliamo che si allontanino troppo dalla lora forma originale. Questo equilibrio è ciò che permette al transfer learning di essere così efficace.

Il Cuore dela Decisione: Il Token CLS
Sintetizzare l'intera frase in un vettore.
BERT è un modello bidirezionale: ogni parola 'vede' tutte le altre. Ma se dobbiamo decidere se l'intera frase è positiva o negativa a chi chiediamo? Serve un rappresentante di classe, questo è il token CLS
Il token CLS (Classification) viene inserito all'inizio di ogni sequenza proprio per accumulare l'informazione globale necessaria alla testa finale. Quindi non porta un significato proprio ma agisce come una spugna, grazie al meccanismo di attenzione, il token CLS assorbe un po' di informazione da ogni passaggio della sequenza, ed alla fine del viaggio, il vettore associato a questo token, è la sintesi perfetta dell'intero messaggio.

Vediamo come estrarre fisicamente questo vettore dal flusso di dati
Gestione dell'Outoput di CLS
Dal pooling alla predizione.
Quando BERT finisce di elaborare ci restituisce un enorme matrice chiamata Hidden State, vettore di dimensione 768 (per la versione base) per ogni token (parola) della sequenza. Il nostro obbiettivo è l'indice 0 di questo token, la prima riga di questa matrice è il nostro token CLS. Spesso applichiamo un processo chiamato Pooling che estrae o trasforma il vettore CLS tramite un layer denso con attivazione 'tanh' prima della classificazione (prende quel CLS e lo rende pronto per il layer decisionale).
La cosa affascinante è che questo vettore CLS non è statico, se cambi anche una sola virgola nella frase il vettore cambia, il suo valore dipende da tutte le parole presenti nella frase, è il risultato del contesto globale.

Ma come trasformiamo questo vettore in una decisione finale?

Layer Lineari Finali
Prendiamo il vettore da 768 e lo proiettiamo sun N neuroni, dove N è il numero di classi del nostro task (mapping dimensionale). I 768 numeri di BERT vengono colassati in una singola etihetta. Ed è qui la verità dove la conoscenza universale del linguaggio si trasforma in una risposta pratica al problema.
Se abbiamo due classi (spam/ham) abbiamo due neuroni in uscita.
Applichiamo il dropout sull'output di CLS prima della proiezione finale per aumentare la robustezza del classificatore (drop-out in fine-tuning).
A differenza dei modelli BoW, qui il gradiente fluisce dalla testa finale indietro attraverso tutti i 12 (o 24) layer di BERT
Non stiamo solo addestrando l'ultimo layer, stiamo chiedendo al gradiente di fluire attraverso tutti i layer di BERT, rifinendo, leggermente, la comprensione del modello,  affinchè diventi sensibile alle sfumature del nostro specifico datasets.

In [1]:
"""
================================================================================================
Fine-tuning di BERT per la Classificazione (Best Practices 2026)
================================================================================================
DESCRIZIONE:
Questo scriptmostra come prendere un modello pre-addestrato (BERT) e "specializzarlo" per un compito
di classificazione del testo (Sentiment Analysis).

INTERAZIONI PRINCIPALI:
1. HUGGING FACE (Libreria Transformers): Fornisce il "corpo" del modello (BertModel) e il 
   "traduttore" (Tokenizer) che trasforma le parole in numeri.
2. KERAS 3: Funge da "regista" (API Funzionale). Definiamo come i dati entrano, come passano 
   attraverso BERT e come arrivano al "decisore finale" (Dense Layer).
3. PYTORCH: Funziona come "motore" sotto il cofano, eseguendo i calcoli matematici richiesti 
   da Keras e BERT.
================================================================================================
"""

import os

os.environ["KERAS_BACKEND"] = "torch"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import torch
import keras

print("Backend Keras:", keras.config.backend())
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])
print("CUDA disponibile:", torch.cuda.is_available())
print("Numero GPU visibili:", torch.cuda.device_count())

import keras
from keras import layers
from transformers import AutoTokenizer, BertModel
import torch
import numpy as np



def get_tokenizer(model_name="bert-base-uncased"):
    """
    Inizializza il 'traduttore' che converte il testo in numeri comprensibili ai neuroni.
    
    PARAMETRI:
        model_name: Il nome del modello BERT standard (es. 'base' non distingue maiuscole).
    
    COM'È FATTO:
        Usa la classe 'AutoTokenizer' che scarica automaticamente le regole di scomposizione 
        delle parole (WordPiece) specifiche per quella versione di BERT.
    """
    print(f"[INFO] Caricamento del tokenizer per {model_name}...")
    return AutoTokenizer.from_pretrained(model_name)

def build_bert_classifier(model_name="bert-base-uncased", num_classes=2):
    """
    Costruisce l'architettura neurale completa: Corpo di BERT + Testa di Classificazione.
    
    Questa funzione usa l'API Funzionale di Keras per creare un "grafo" di calcolo.
    """
    
    # -------------------------------------------------------------------------
    # 1. DEFINIZIONE DEGLI INGRESSI (Input Layer)
    # -------------------------------------------------------------------------
    # BERT non riceve testo, ma due flussi di numeri:
    input_ids = layers.Input(shape=(None,), dtype="int32", name="input_ids") # Gli ID delle parole
    attention_mask = layers.Input(shape=(None,), dtype="int32", name="attention_mask") # 1 se è parola, 0 se è spazio vuoto (padding)

    # -------------------------------------------------------------------------
    # 2. IL "CORPO" DI BERT (Encoder)
    # -------------------------------------------------------------------------
    # Carichiamo i pesi già addestrati su miliardi di frasi dal web.
    bert_body = BertModel.from_pretrained(model_name)
    
    # Creiamo un layer Keras personalizzato per "incapsulare" BERT.
    # Questo serve perché BERT (PyTorch) deve comunicare correttamente con Keras.
    class BertLayer(keras.layers.Layer):
        def __init__(self, model, **kwargs):
            super().__init__(**kwargs)
            self.bert = model # Il modello Hugging Face viene salvato qui dentro

        def call(self, inputs):
            # Ingressi: [input_ids, attention_mask]
            # Uscita: estraiamo 'pooler_output', ovvero il riassunto del token [CLS] (Slide 11-14)
            outputs = self.bert(input_ids=inputs[0], attention_mask=inputs[1])
            return outputs.pooler_output

    # Applichiamo il layer di BERT ai nostri ingressi
    # cls_representation è un vettore di 768 numeri che "riassume" l'intera frase
    cls_representation = BertLayer(bert_body)([input_ids, attention_mask])

    # -------------------------------------------------------------------------
    # 3. LA "TESTA" DECISIONALE (Classification Head)
    # -------------------------------------------------------------------------
    # Aggiungiamo uno strato di Dropout per evitare che il modello impari a memoria (overfitting)
    x = layers.Dropout(0.1)(cls_representation)
    
    # Lo strato Denso finale trasforma i 768 numeri di BERT nelle probabilità delle nostre classi.
    # Se num_classes=2 (Sentiment), avremo 2 neuroni in uscita.
    output = layers.Dense(num_classes, activation="softmax", name="classifier")(x)

    # -------------------------------------------------------------------------
    # 4. ASSEMBLAGGIO E COMPILAZIONE
    # -------------------------------------------------------------------------
    # Uniamo ingressi e uscite in un unico oggetto Modello
    model = keras.Model(inputs=[input_ids, attention_mask], outputs=output)

    # L'ottimizzatore AdamW è fondamentale nel fine-tuning (Slide 10)
    # Usiamo un learning_rate microscopico (2e-5) per non cancellare la memoria di BERT.
    optimizer = keras.optimizers.AdamW(learning_rate=2e-5, weight_decay=0.01)
    
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy", # Ottima per etichette intere (0, 1, 2...)
        metrics=["accuracy"]
    )
    
    return model

def prepare_dummy_data(tokenizer, texts, labels):
    """
    Trasforma le frasi umane in tensori (matrici di numeri) pronti per essere elaborati.
    
    INTERAZIONE:
        Il tokenizer crea 'input_ids' e 'attention_mask' automaticamente.
    """
    # Padding=True: rende tutte le frasi lunghe uguali aggiungendo zeri
    # Truncation=True: taglia le frasi troppo lunghe (oltre i 512 token)
    encodings = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        return_tensors="pt" # Restituisce tensori PyTorch
    )
    
    # Distribuiamo i dati in un dizionario che Keras capisce al volo
    x = {
        "input_ids": encodings["input_ids"].numpy(),
        "attention_mask": encodings["attention_mask"].numpy()
    }
    y = np.array(labels) # Le etichette (0 o 1) diventano un array NumPy
    
    return x, y

# ================================================================================================
# AVVIO DEL PROCESSO (MAIN)
# ================================================================================================

if __name__ == "__main__":
    print("\n--- INIZIO LEZIONE PRATICA: FINE-TUNING DI BERT ---")

    # 1. SELEZIONE DEL MODELLO
    # 'bert-base-uncased' è la versione standard (12 layer, 768 neuroni per layer).
    CHECKPOINT = "bert-base-uncased"
    
    # 2. PREPARAZIONE TOKENIZER
    # Trasforma "Ciao" -> [101, 2345, 102]
    tokenizer = get_tokenizer(CHECKPOINT)
    
    # 3. CREAZIONE DATASET DI ESEMPIO (Sentiment Analysis Mini)
    # Immaginiamo di voler classificare se un commento è positivo (1) o negativo (0)
    texts_example = [
        "Incredibile! Questa lezione è chiarissima e utilissima.",   # 1 (Positivo)
        "Purtroppo non ho capito nulla, il codice è troppo difficile.", # 0 (Negativo)
        "Il token [CLS] è fondamentale per capire l'intera frase."     # 1 (Positivo)
    ]
    labels_example = [1, 0, 1]
    
    # Trasformiamo i testi in numeri
    x_train, y_train = prepare_dummy_data(tokenizer, texts_example, labels_example)
    
    # 4. COSTRUZIONE DEL MODELLO
    # Qui avviene la magia: carichiamo BERT e ci montiamo sopra la nostra "testa"
    model = build_bert_classifier(CHECKPOINT, num_classes=2)
    
    # Mostriamo a video lo schema del modello (L'architettura funzionale)
    model.summary()

    # 5. TRAINING (Il cuore del Fine-tuning)
    # Iniziamo a regolare i pesi. Con BERT bastano pochissime epoche (Slide 8).
    print("\n[STEP] Avvio dell'addestramento su Keras con backend PyTorch...")
    model.fit(
        x_train, 
        y_train, 
        epochs=3,      # Passiamo sul dataset 3 volte
        batch_size=2   # Elaboriamo 2 frasi alla volta per non saturare la memoria
    )
    
    # 6. TEST DI PREDIZIONE
    # Proviamo con una frase mai vista prima dal modello
    new_comment = ["Questo approccio integrato Keras-BERT è il futuro!"]
    x_test, _ = prepare_dummy_data(tokenizer, new_comment, [0])
    
    print(f"\n[TEST] Analisi della frase: '{new_comment[0]}'")
    prediction = model.predict(x_test)
    
    # Il risultato è una probabilità: [prob_negativo, prob_positivo]
    print(f"Probabilità (Negativo vs Positivo): {prediction[0]}")
    classe_predetta = np.argmax(prediction)
    print(f"Risultato: {'POSITIVO' if classe_predetta == 1 else 'NEGATIVO'}")

    print("\n--- FINE ESEMPIO ---")

Backend Keras: torch
CUDA_VISIBLE_DEVICES: -1
CUDA disponibile: False
Numero GPU visibili: 0


c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



--- INIZIO LEZIONE PRATICA: FINE-TUNING DI BERT ---
[INFO] Caricamento del tokenizer per bert-base-uncased...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5643.64it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_ids           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_mask      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_layer          │ (None, 768)       │ 109,482,2… │ input_ids[0][0],  │
│ (BertLayer)         │                   │            │ attention_mask[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 768)       │          0 │ bert_layer[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classifier (Dense)  │ (None, 2)         │      1,538 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 109,483,778 (417.65 MB)

 Trainable params: 109,483,778 (417.65 MB)

 Non-trainable params: 0 (0.00 B)


[STEP] Avvio dell'addestramento su Keras con backend PyTorch...
Epoch 1/3
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 720ms/step - accuracy: 0.0000e+00 - loss: 1.3932
Epoch 2/3
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 478ms/step - accuracy: 0.6667 - loss: 0.5310
Epoch 3/3
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 479ms/step - accuracy: 1.0000 - loss: 0.3237

[TEST] Analisi della frase: 'Questo approccio integrato Keras-BERT è il futuro!'
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Probabilità (Negativo vs Positivo): [0.21319835 0.7868017 ]
Risultato: POSITIVO

--- FINE ESEMPIO ---
